# Học tổ hợp: Bagging và Random Forest

*Bài 9, mục 9.2.1.*

## 1. Vì sao kết hợp nhiều mô hình lại tốt hơn?

Giả sử có $T$ bộ phân loại **độc lập**, mỗi bộ đúng với xác suất $p > 0.5$. Bỏ phiếu đa số đúng khi có hơn một nửa số bộ đúng; số bộ đúng tuân theo phân phối nhị thức:
$$P(\text{đa số đúng}) = \sum_{k > T/2}\binom{T}{k}p^k(1-p)^{T-k}$$

In [1]:
from math import comb

import numpy as np


def majority_vote_accuracy(T, p):
    return sum(comb(T, k) * p**k * (1 - p)**(T - k) for k in range(T // 2 + 1, T + 1))


for T in [1, 3, 5, 11, 25, 101]:
    print('T = %3d bộ phân loại, mỗi bộ đúng 70%%: bỏ phiếu đúng %.4f' % (T, majority_vote_accuracy(T, 0.7)))

T =   1 bộ phân loại, mỗi bộ đúng 70%: bỏ phiếu đúng 0.7000
T =   3 bộ phân loại, mỗi bộ đúng 70%: bỏ phiếu đúng 0.7840
T =   5 bộ phân loại, mỗi bộ đúng 70%: bỏ phiếu đúng 0.8369
T =  11 bộ phân loại, mỗi bộ đúng 70%: bỏ phiếu đúng 0.9218
T =  25 bộ phân loại, mỗi bộ đúng 70%: bỏ phiếu đúng 0.9825
T = 101 bộ phân loại, mỗi bộ đúng 70%: bỏ phiếu đúng 1.0000


Điều kiện then chốt là các bộ phân loại **mắc lỗi khác nhau** (độc lập, đa dạng). Nếu chúng giống hệt nhau thì bỏ phiếu không cải thiện gì.

## 2. Bootstrap

Mẫu bootstrap: lấy ngẫu nhiên **có hoàn lại** $N$ mẫu từ tập $N$ mẫu. Trung bình mỗi mẫu bootstrap chỉ chứa khoảng $1 - (1 - 1/N)^N \approx 1 - e^{-1} \approx 63.2\%$ số mẫu khác nhau; phần còn lại (khoảng 36.8%) gọi là mẫu **out-of-bag** (OOB).

In [2]:
rng = np.random.default_rng(0)
N = 1000
for trial in range(3):
    idx = rng.integers(0, N, size=N)         # chọn N chỉ số có hoàn lại
    print('Mẫu bootstrap %d: %d mẫu khác nhau (%.1f%%)' % (trial + 1, len(np.unique(idx)), 100 * len(np.unique(idx)) / N))
print('Lý thuyết: 1 - (1 - 1/N)^N = %.4f' % (1 - (1 - 1 / N)**N))

Mẫu bootstrap 1: 622 mẫu khác nhau (62.2%)
Mẫu bootstrap 2: 641 mẫu khác nhau (64.1%)
Mẫu bootstrap 3: 636 mẫu khác nhau (63.6%)
Lý thuyết: 1 - (1 - 1/N)^N = 0.6323


## 3. Cây quyết định có phương sai cao

Dữ liệu ung thư vú (Bài 7). Huấn luyện cây đầy đủ trên các mẫu bootstrap khác nhau: các cây cho dự đoán khác nhau đáng kể trên cùng tập kiểm tra.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

from sklearn.tree import DecisionTreeClassifier

predictions = []
for b in range(5):
    idx = rng.integers(0, len(X_train), size=len(X_train))
    tree = DecisionTreeClassifier(random_state=b).fit(X_train[idx], y_train[idx])
    predictions.append(tree.predict(X_test))
    print('Cây %d: độ chính xác kiểm tra %.4f' % (b + 1, tree.score(X_test, y_test)))
predictions = np.array(predictions)
disagree = (predictions != predictions[0]).any(axis=0).mean()
print('Tỉ lệ mẫu kiểm tra mà 5 cây không thống nhất: %.1f%%' % (100 * disagree))
vote = (predictions.mean(axis=0) >= 0.5).astype(int)
print('Bỏ phiếu đa số của 5 cây: %.4f' % (vote == y_test).mean())

Cây 1: độ chính xác kiểm tra 0.9064
Cây 2: độ chính xác kiểm tra 0.9181
Cây 3: độ chính xác kiểm tra 0.9123
Cây 4: độ chính xác kiểm tra 0.9357
Cây 5: độ chính xác kiểm tra 0.9064
Tỉ lệ mẫu kiểm tra mà 5 cây không thống nhất: 14.6%
Bỏ phiếu đa số của 5 cây: 0.9240


## 4. Bagging

**Bagging** (Bootstrap AGGregatING): huấn luyện $T$ mô hình trên $T$ mẫu bootstrap, kết hợp bằng bỏ phiếu (phân loại) hoặc lấy trung bình (hồi quy).

In [4]:
from sklearn.ensemble import BaggingClassifier

single_tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
bagging = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=0)
bagging.fit(X_train, y_train)
print('Một cây quyết định : kiểm tra %.4f' % single_tree.score(X_test, y_test))
print('Bagging 100 cây    : kiểm tra %.4f' % bagging.score(X_test, y_test))
print('Điểm OOB của Bagging: %.4f (ước lượng độ chính xác không cần tập kiểm tra)' % bagging.oob_score_)

Một cây quyết định : kiểm tra 0.9064
Bagging 100 cây    : kiểm tra 0.9357
Điểm OOB của Bagging: 0.9548 (ước lượng độ chính xác không cần tập kiểm tra)


## 5. Random Forest

Random Forest = Bagging các cây quyết định + **chọn ngẫu nhiên một tập con đặc trưng tại mỗi nút** (thường $\sqrt{d}$ đặc trưng với phân loại). Việc này làm các cây khác nhau hơn (giảm tương quan), nên bỏ phiếu hiệu quả hơn.

In [5]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=100, max_features='sqrt', oob_score=True, random_state=0, n_jobs=-1)
forest.fit(X_train, y_train)
print('Random Forest 100 cây: kiểm tra %.4f, OOB %.4f' % (forest.score(X_test, y_test), forest.oob_score_))

Random Forest 100 cây: kiểm tra 0.9532, OOB 0.9698


Độ quan trọng của đặc trưng: trung bình mức giảm độ vẩn đục (Gini) do mỗi đặc trưng đóng góp trên mọi cây.

In [6]:
names = load_breast_cancer().feature_names
order = np.argsort(forest.feature_importances_)[::-1][:8]
for i in order:
    print('%-22s %.3f' % (names[i], forest.feature_importances_[i]))

worst perimeter        0.171
worst radius           0.128
worst concave points   0.108
mean concave points    0.094
mean perimeter         0.076
mean concavity         0.068
worst area             0.066
mean area              0.049


## 6. Số cây trong rừng

In [7]:
for n in [1, 5, 10, 50, 100, 300]:
    model = RandomForestClassifier(n_estimators=n, random_state=0, n_jobs=-1).fit(X_train, y_train)
    print('n_estimators = %3d: kiểm tra %.4f' % (n, model.score(X_test, y_test)))

n_estimators =   1: kiểm tra 0.8889
n_estimators =   5: kiểm tra 0.9415


n_estimators =  10: kiểm tra 0.9357


n_estimators =  50: kiểm tra 0.9474


n_estimators = 100: kiểm tra 0.9532


n_estimators = 300: kiểm tra 0.9532
